In [1]:
pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# This dictionary stores all the scenes in your story. 
# "start" is the default first scene.
story_data = {
    "start": {
        "text": "You wake up in a dark, mysterious forest. Two paths diverge in front of you. One looks well-trodden, the other is overgrown with thorns.",
        "choices": {
            "Take the well-trodden path": "clear_path",
            "Take the thorny path": "thorny_path"
        }
    }
}

print("Story engine initialized! Run the next cells to build and play.")

Story engine initialized! Run the next cells to build and play.


In [3]:
# Create UI Widgets for the editor
creator_out = widgets.Output()

html_title = widgets.HTML("<h2>📜 Story Creator</h2><p>Create or overwrite a scene (node). Use the exact <b>Node ID</b> you used as a destination in previous choices.</p>")

node_id_input = widgets.Text(description="Node ID:", placeholder="e.g., clear_path")
story_text_input = widgets.Textarea(description="Story Text:", layout=widgets.Layout(width='80%', height='100px'))

# Choice 1
c1_text = widgets.Text(description="Choice 1:", placeholder="Text the player sees")
c1_dest = widgets.Text(description="Dest 1:", placeholder="Node ID this leads to")

# Choice 2
c2_text = widgets.Text(description="Choice 2:", placeholder="Text the player sees")
c2_dest = widgets.Text(description="Dest 2:", placeholder="Node ID this leads to")

save_btn = widgets.Button(description="Save Scene", button_style='success', icon='save')

def save_node(b):
    new_choices = {}
    if c1_text.value and c1_dest.value:
        new_choices[c1_text.value] = c1_dest.value
    if c2_text.value and c2_dest.value:
        new_choices[c2_text.value] = c2_dest.value
        
    # Save to our global dictionary
    story_data[node_id_input.value] = {
        "text": story_text_input.value,
        "choices": new_choices
    }
    
    with creator_out:
        clear_output()
        print(f"✅ Scene '{node_id_input.value}' saved successfully!")
        print(f"Current scenes in memory: {list(story_data.keys())}")
        
        # Clear inputs for the next scene
        node_id_input.value = ""
        story_text_input.value = ""
        c1_text.value = ""
        c1_dest.value = ""
        c2_text.value = ""
        c2_dest.value = ""

save_btn.on_click(save_node)

# Display the editor interface
editor_ui = widgets.VBox([
    html_title,
    node_id_input,
    story_text_input,
    widgets.HTML("<b>Add Choices (Leave blank for an ending scene):</b>"),
    widgets.HBox([c1_text, c1_dest]),
    widgets.HBox([c2_text, c2_dest]),
    save_btn,
    creator_out
])

display(editor_ui)

In [4]:
player_out = widgets.Output()

def play_scene(node_id):
    with player_out:
        clear_output()
        
        # Check if the player wandered off the map
        if node_id not in story_data:
            display(widgets.HTML(f"<h3 style='color:red;'>Error 404: Scene Not Found</h3>"))
            print(f"The node '{node_id}' hasn't been created yet.")
            print("Go back to the Story Creator to write it!")
            return
            
        node = story_data[node_id]
        
        # Display the story text
        display(widgets.HTML(f"<p style='font-size:16px; line-height:1.5;'>{node['text']}</p>"))
        display(widgets.HTML("<hr>"))
        
        # If there are no choices, it's an ending
        if not node.get("choices"):
            display(widgets.HTML("<b>-- THE END --</b>"))
            restart_btn = widgets.Button(description="Play Again", button_style='info', icon='refresh')
            restart_btn.on_click(lambda b: play_scene("start"))
            display(restart_btn)
            return
            
        # Generate buttons for choices
        buttons = []
        for choice_text, next_node in node["choices"].items():
            btn = widgets.Button(description=choice_text, layout=widgets.Layout(width='auto', min_width='200px'))
            
            # Use a closure to capture the correct next_node for each button
            def make_callback(destination):
                return lambda b: play_scene(destination)
                
            btn.on_click(make_callback(next_node))
            buttons.append(btn)
            
        display(widgets.VBox(buttons))

# Start the game
display(player_out)
play_scene("start")

Output()